# Phase 6d Wave 1: Why Quarks Are Locked Inside Hadrons (And What It Has to Do With Perfect Fluids)

**A non-specialist tour of confinement, the Polyakov loop, Svetitsky-Yaffe universality, and the KSS viscosity bound.**

Companion notebook to `papers/paper36_center_symmetry/paper_draft.tex`.

## A famous open problem

Quark confinement is one of the seven Clay Millennium Prize problems (under "Yang-Mills mass gap"). It asks: why are quarks always trapped inside neutron-like and meson-like hadrons, never seen alone? In QCD, the strong force grows *stronger* with distance — pull two quarks apart and the energy in the connecting flux tube rises until you create new quark-antiquark pairs. Quarks come out only as part of color-neutral combinations.

**The modern reframing.** Confinement = unbreaking of a *higher-form symmetry* called the ℤ_N **center symmetry** of SU(N) gauge theory. The order parameter is the **Polyakov loop** $P$ — a spatial average of the time-circulation of the gauge field. At low temperature, $\langle P \rangle = 0$ (confined). At high temperature, $\langle P \rangle \neq 0$ (deconfined plasma).

## The Svetitsky-Yaffe miracle

Svetitsky and Yaffe (1982) proved that the deconfinement transition in 4D SU(N) gauge theory is in the same *universality class* as a 3D ℤ_N spin model:

- SU(2) deconfinement → 3D Ising magnet ($\nu \approx 0.6299$)
- SU(3) deconfinement → 3D 3-state Potts model ($\nu \approx 0.5$)

This means: heavy-ion experiments (RHIC, LHC) measuring the SU(3) deconfinement transition can be interpreted using statistical-mechanics tools developed for the 3-state Potts ferromagnet. The microscopic theories are completely different; the *symmetry-breaking pattern* is the same.

## The KSS perfect-fluid bound

The quark-gluon plasma at RHIC turned out to be the most *perfect* fluid known: shear-viscosity-to-entropy ratio $\eta/s \approx 0.1\text{–}0.2$, only $\sim 1.3\text{–}2.5\times$ the universal lower bound from string theory:

$$\frac{\eta}{s} \geq \frac{1}{4\pi} \approx 0.0796.$$

This is the KSS bound (Kovtun-Son-Starinets 2005). It came from gauge/gravity duality (string theory) and the experimental confirmation was a stunning surprise.

## What this paper adds

We formalize the confinement-as-center-symmetry framing in Lean 4, prove the Polyakov-loop biconditional ($P = 0$ iff confining, fixed by the center action), encode the Svetitsky-Yaffe map and its critical exponents, and propose a **higher-form-symmetry topological-defect transport correction** — building on Hofman-Iqbal 2018 generalized hydrodynamics — that predicts $\eta/s \in [\mathrm{KSS}, 2\cdot\mathrm{KSS}]$ — a falsifiable narrow window with two concrete falsifiers ($\eta/s = 0$ and $\eta/s = 1$ both ruled out).

## 1. The center generator $\zeta_N = e^{2\pi i / N}$

Concretely:
- $\zeta_2 = e^{i\pi} = -1$ (SU(2))
- $\zeta_3 = e^{2\pi i / 3} = -1/2 + i\sqrt{3}/2$ (SU(3))

These are roots of unity ($\zeta_N^N = 1$, $|\zeta_N| = 1$). They act on the Polyakov loop by multiplication: $P \mapsto \zeta_N \cdot P$. Only $P = 0$ is fixed by all such actions — the confining vacuum is the *unique* center-invariant configuration.

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.center_symmetry import (
    CenterZN, center_phase, center_action,
    confining, polyakov_magnitude,
    UniversalityClass, svetitsky_yaffe_class, critical_exponent_nu,
    KSS_BOUND, WalkerWangPrediction,
)
from src.core.visualizations import COLORS, fig_polyakov_loop_deconfinement

for N in [2, 3, 4]:
    z = CenterZN(N)
    zeta = center_phase(z)
    print(f'  ζ_{N} = e^(2πi/{N}) = {zeta.real:+.4f} + {zeta.imag:+.4f}i    |ζ| = {abs(zeta):.4f}')

  ζ_2 = e^(2πi/2) = -1.0000 + +0.0000i    |ζ| = 1.0000
  ζ_3 = e^(2πi/3) = -0.5000 + +0.8660i    |ζ| = 1.0000
  ζ_4 = e^(2πi/4) = +0.0000 + +1.0000i    |ζ| = 1.0000


## 2. Polyakov-loop and the confinement order parameter

Confinement is the statement that $|\langle P \rangle| = 0$ — the Polyakov loop has zero expectation value. Equivalently: $P = 0$ is the only point fixed by the center action. Try a few concrete cases.

In [2]:
Z3 = CenterZN(3)
for label, P in [
    ('Confining vacuum',                  0+0j),
    ('Deconfined real positive',          1+0j),
    ('Deconfined: 60° in complex plane',  0.5+0.866j),
]:
    P_acted = center_action(Z3, P)
    print(f'  {label:35s}  |P| = {polyakov_magnitude(P):.4f}   confining = {confining(P)}')
    print(f'    ζ_3 · P = {P_acted.real:+.4f}{P_acted.imag:+.4f}j     fixed by ζ_3: {abs(P_acted - P) < 1e-10}')
    print()

  Confining vacuum                     |P| = 0.0000   confining = True
    ζ_3 · P = -0.0000+0.0000j     fixed by ζ_3: True

  Deconfined real positive             |P| = 1.0000   confining = False
    ζ_3 · P = -0.5000+0.8660j     fixed by ζ_3: False

  Deconfined: 60° in complex plane     |P| = 1.0000   confining = False
    ζ_3 · P = -1.0000+0.0000j     fixed by ζ_3: False



## 3. Svetitsky-Yaffe universality classes

The deconfinement critical exponents that *would be measured* in a careful lattice / heavy-ion analysis match the corresponding ℤ_N spin model. The 3D Ising critical exponents have been computed to 4-decimal precision via conformal-bootstrap methods (Kos-Poland-Simmons-Duffin 2016).

In [3]:
for N in [2, 3]:
    z = CenterZN(N)
    uc = svetitsky_yaffe_class(z)
    nu = critical_exponent_nu(uc)
    print(f'  SU({N}) deconfinement → {uc.name:25s}  ν = {nu}')

nu_ising = critical_exponent_nu(UniversalityClass.ISING)
nu_potts = critical_exponent_nu(UniversalityClass.THREE_STATE_POTTS)
print()
print(f'  Quantitative SU(2)-SU(3) discriminator: ν_Ising = {nu_ising}, ν_Potts = {nu_potts}')
print(f'  ν_Ising > ν_Potts: {nu_ising > nu_potts}    (load-bearing comparison; threshold-free)')

  SU(2) deconfinement → ISING                      ν = 0.6299
  SU(3) deconfinement → THREE_STATE_POTTS          ν = 0.5

  Quantitative SU(2)-SU(3) discriminator: ν_Ising = 0.6299, ν_Potts = 0.5
  ν_Ising > ν_Potts: True    (load-bearing comparison; threshold-free)


## 4. The KSS bound and the Walker-Wang prediction

The famous KSS bound $\eta/s \geq 1/(4\pi) \approx 0.0796$. Quark-gluon plasma sits 1.5–2× this value. The paper proposes a **Walker-Wang topological-defect transport correction** that says $\eta/s$ should sit in a narrow window $[\mathrm{KSS}, 2 \cdot \mathrm{KSS}]$ — close to the bound but not below it. Two values are concretely ruled out by Lean falsifiers: $\eta/s = 0$ (below the bound) and $\eta/s = 1$ (well above the upper).

In [ ]:
print(f'  KSS bound 1/(4π)            = {KSS_BOUND:.6f}')
print(f'  Walker-Wang window upper    = {2 * KSS_BOUND:.6f}')
print(f'  RHIC quark-gluon plasma     ≈ 0.10–0.20 (~1.3–2.5× KSS)')
print(f'    (Heinz-Snellings 2013 review, Annu. Rev. Nucl. Part. Sci. 63:123,')
print(f'     arXiv:1301.2826; constraints from PHENIX + STAR azimuthal-anisotropy fits)')
print()
for label, eta in [
    ('falsifier 1: zero',    0.0),
    ('KSS lower (witness)',  KSS_BOUND),
    ('within window: 1.5KSS', 1.5 * KSS_BOUND),
    ('upper bound: 2·KSS',   2 * KSS_BOUND),
    ('falsifier 2: unity',   1.0),
]:
    pred = WalkerWangPrediction(eta)
    print(f'  {label:30s}  η/s = {eta:.5f}   consistent with prediction = {pred.consistent}')

## 5. Visualization

**Left panel.** $|P|$ vs $T/T_c$ for SU(2) (Ising, blue) and SU(3) (Potts, amber). Below $T_c$: $|P| = 0$ — confining. Above $T_c$: $|P| \propto (T/T_c - 1)^\beta$, with the universality-class-specific exponent $\beta$. The two curves rise differently above $T_c$ — the *quantitative* discriminator that universality-class measurements would test.

**Right panel.** $\eta/s$ on a log-scale bar chart, with horizontal reference lines at the KSS bound (red solid) and Walker-Wang upper (blue dotted). Bars in the [KSS, 2·KSS] window are blue; bars outside (the two falsifiers and the trivial cases) are amber. The two falsifier bars (η/s = 0, η/s = 1) are clearly outside the window.

In [5]:
# viz-ref: fig_polyakov_loop_deconfinement
fig_polyakov_loop_deconfinement().show()

## 6. Honest scope

**What this work proves.** The center-symmetry / confinement framing is encoded as 18 machine-checked Lean theorems with zero `sorry` and zero new axioms. The Polyakov-loop biconditional, the Svetitsky-Yaffe universality classes with literature-anchored exponents, and the KSS bound bracket [0.07, 0.08] are all formalized.

**What this work does not prove.**

1. *That confinement happens at all.* We characterize confinement structurally; we do not derive it from QCD dynamics. (That's the actual Millennium Prize problem.)
2. *That Walker-Wang transport corrections operate in nature.* The $[\mathrm{KSS}, 2 \cdot \mathrm{KSS}]$ window is a tracked Prop — a testable prediction, not a derived consequence. HPC validation in Phase 6B is gated.
3. *That RHIC measurements specifically test Walker-Wang.* Heavy-ion $\eta/s$ values cluster at 0.1–0.2, comfortably within the window, but distinguishing Walker-Wang specifically from generic strong-coupling holography requires sub-leading transport coefficient measurements.

**What would falsify the bridge.**

1. A measurement of $\eta/s < \mathrm{KSS}$ in any strongly-coupled system — would violate the KSS bound itself (the bound has held in every system measured so far).
2. A reliable measurement of $\eta/s$ in heavy-ion systems above $2 \cdot \mathrm{KSS} \approx 0.16$ with sub-leading transport coefficients consistent with topological-defect corrections — would invalidate the Walker-Wang window and force a wider tolerance.
3. A demonstration that Svetitsky-Yaffe universality fails in a 4D gauge theory whose center symmetry is unambiguous — would invalidate the structural map.

**Why this matters strategically.** The center-symmetry framing makes confinement a *structural* property of the gauge group, not a dynamical mystery. Combined with Walker-Wang, it gives the project's gravity / dark-sector framework a concrete bridge to QCD phenomenology — the same anyon-spectrum machinery used for horizon entropy (Phase 6a) and Hayden-Preskill scrambling (Phase 6c W4) here applies to transport in the QGP.

## 7. Where to go next

- **Paper:** `papers/paper36_center_symmetry/paper_draft.tex` — full center-symmetry formalization.
- **Lean module:** `lean/SKEFTHawking/CenterSymmetryConfinement.lean` — 18 theorems.
- **Companion technical notebook:** `Phase6d1_CenterSymmetry_Technical.ipynb` — paper-section walkthrough with full Lean inventory.
- **Next waves:**
  - Phase 6d Wave 2 (`paper37_chiral_ssb`): chiral symmetry breaking and the GMOR relation — pairs with this wave (both are QCD-physics formalizations).
  - Phase 6d Wave 3 (`paper38_cfl`): the color-flavor-locked dense-matter phase, with an emergent ℤ_3 that *coincides* with the QCD center ℤ_3 — the Phase 6d correctness-push anchor.